# Renewables in Electricity Markets: System Perspective

This is the solution to Assignment 1 proposed in the course "Renewables in Electricity Markets" from DTU in 2025.

The chosen system is the IEEE 24-bus test reliability system, shown in the figure below. Data is in the file IEEE 24-bus reliability test system - data.xlsx. 

# ![System](img/System.png)

## Step 6: Reserve Market

As in step 5, we will simulate a copper plate model in a single hour, but this time to model the reserve market. The chosen hour will be 8 AM. We will first clear the reserve market, where offers submitted by generators will simulate that they are willing to offer reserve at a price equal to what they expect to earn in the day ahead market.

### Input data

In [130]:
# Import the libraries to be used
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import linopy
np.random.seed(42)

#### Read data

In [131]:
# system data
data = pd.read_excel('data/IEEE 24-bus reliability test system - data.xlsx', sheet_name=None)
print(data.keys())

dict_keys(['Generators', 'Costs', 'SystemLoadProfile', 'LoadDist', 'BidTypes', 'Lines', 'WindFarms', 'LineModification'])


In [132]:
#dict_keys(['Generators', 'Costs', 'SystemLoadProfile', 'LoadDist', 'BidTypes', 'Lines', 'WindFarms', 'LineModification'])
generators = data['Generators']
costs = data['Costs']
load_profile = data['SystemLoadProfile']
load_dist = data['LoadDist']
load_bid_types = data['BidTypes']
lines = data['Lines']
wind_farms = data['WindFarms']
line_modification = data['LineModification']

In [133]:
# windfarm data
wind_production_dk = pd.read_csv('data/ninja_wind_55.4104_12.3039_corrected.csv', skiprows=3)

#### Handle demand data

In [134]:
# Assign demand to each bus based on the load distribution
demand = load_profile.merge(load_dist, how='cross')
demand['Demand_MW'] = demand['System_demand_MW'] * demand['Percent_of_system_load'] / 100
demand['Hour'] = demand['Hour'] -1  # Adjust hour to be 0-indexed
demand.drop(columns=['System_demand_MW', 'Percent_of_system_load'], inplace=True)

In [135]:
# create demand bids: Load (id), Node (id), Quantity (MW), Price ($/MWh)
demand_bid = demand.merge(load_bid_types, left_on='Load_type', right_on='Load_type', how='left')
demand_bid['Quantity_MW'] = demand_bid['Demand_MW']*demand_bid['Quantity_perc']/100
demand_bid = demand_bid[['Bid', 'Load', 'Node', 'Hour', 'Load_type', 'Quantity_MW', 'Price_$/MWh']]
demand_bid['Load'] = 'L' + demand_bid['Load'].astype(str) 

#### Handle WindFarm data

In [136]:
#select 6 random days to simulate the wind production of each generator
np.random.seed(42)
wind_production_dk['local_time'] = pd.to_datetime(wind_production_dk['local_time'])
wind_production_dk['day'] = wind_production_dk['local_time'].dt.date
random_days = np.random.choice(wind_production_dk['day'].unique(), size=len(wind_farms), replace=False)
wind_production_sample = wind_production_dk[wind_production_dk['day'].isin(random_days)]
# assign one day to one zone windfarm
wind_production_sample['WindFarm'] = wind_production_sample['day'].apply(lambda x: random_days.tolist().index(x)+1)
wind_production_sample['Hour'] = wind_production_sample['local_time'].dt.hour

In [137]:
wind_forecast_generation = wind_production_sample.merge(wind_farms, on='WindFarm', how='left')
wind_forecast_generation['Generation_MW'] = wind_forecast_generation['Installed_Capacity_MW'] * wind_forecast_generation['electricity']
wind_forecast_generation = wind_forecast_generation[['WindFarm', 'Node', 'Hour', 'Generation_MW']].sort_values(['WindFarm', 'Hour'])
wind_forecast_generation['WindFarm'] = 'W' + wind_forecast_generation['WindFarm'].astype(str)
wind_forecast_generation.rename(columns={'Generation_MW': 'PForecast_MW', 'WindFarm': 'Unit'}, inplace=True)

#### Handle supply data

In [138]:
gen_constraints = generators.merge(costs, on='Unit')
gen_constraints['Unit'] = 'G' + gen_constraints['Unit'].astype(str)

### Modelling

#### Model 1. Expected Daty Ahead Market

To offer their price for reserve, generators need to guess their oportunity cost. I guess their model is more complex, but for simplicity I will assume that they use as an input the expected market price and add a premium (I'm assuming 3% of their variable cost). 

Then their offer price will be:
$$\mu_{i}^{up} = \max(0, \lambda^{DA} - c_{i}) + 3\% \cdot c_i $$
$$\mu_{i}^{down} = \max(0, c_{i} - \lambda^{DA}) + 3\% \cdot c_i $$

Where $\lambda^{DA}$ is the expected market price, $c_i$ is the marginal cost of generator $i$, and $\mu_{i}^{up}$ and $\mu_{i}^{down}$ are the offer prices for up and down regulation, respectively.

In [139]:
# Sets
generators_set = pd.Index(gen_constraints['Unit'].unique(), name='Unit')
wind_farms_set = pd.Index(wind_forecast_generation['Unit'].unique(), name='Unit')
loads_set = pd.Index(demand_bid['Load'].unique(), name='Load')
bids_set = pd.Index(demand_bid['Bid'].unique(), name='Bid')

In [140]:
# Parameters
gen_constraints_xr = gen_constraints.set_index('Unit').to_xarray()
demand_bid_xr = demand_bid.set_index(['Load','Bid','Hour']).to_xarray()
wind_forecast_xr = wind_forecast_generation.set_index(['Unit','Hour']).to_xarray()

# select only hour 8
demand_bid_xr = demand_bid_xr.sel(Hour=8) 
wind_forecast_xr = wind_forecast_xr.sel(Hour=8) 


pmax = gen_constraints_xr['Pmax_MW'] # Maximum power output of each generator in MW
ci = gen_constraints_xr['Ci_$/MWh'] # Variable cost of each generator in $/MWh

wind_forecast = wind_forecast_xr['PForecast_MW'] # Forecasted power output of each wind farm in MW

demand_quantity = demand_bid_xr['Quantity_MW'].reindex(
    Load=loads_set, 
    Bid=bids_set, 
).fillna(0)

demand_price = demand_bid_xr['Price_$/MWh'].reindex(
    Load=loads_set, 
    Bid=bids_set,
).fillna(0)

In [141]:
model = linopy.Model()

# Decision variables: generation
g = model.add_variables(
    coords=[generators_set],
    name="dispatch"
)

# Decision variables: wind generation
w = model.add_variables(
    lower=0,
    coords=[wind_farms_set],
    name="wind_generation"
)

# Decision variables: demand
d = model.add_variables(
    lower=0,
    coords=[loads_set, bids_set],
    name="demand"
)

# Generators capacity constraints
capacity_constraints = model.add_constraints(
    g <= pmax,
    name="capacity_limit"
)

min_generation_constraint = model.add_constraints(
    g >= 0,
    name="min_capacity_limit"
)

# Wind generation constraints
wind_constraints = model.add_constraints(
    w<= wind_forecast,
    name="wind_dispatch"
)

# Demand constraints
demand_constraints = model.add_constraints(
    d <= demand_quantity,
    name="demand_limit"
)

# Power balance constraint
balance = model.add_constraints(
    g.sum(dim="Unit") + w.sum(dim="Unit") == d.sum(dim=["Load","Bid"]),
    name="power_balance"
)

# Then use these in your objective
model.add_objective(
    (g * ci).sum() - (d * demand_price).sum(),
    sense="min"
)

model.solve(solver_name="highs")

Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-vjublopw has 99 rows; 86 cols; 184 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [5e+00, 2e+03]
  Bound   [0e+00, 0e+00]
  RHS     [3e+00, 6e+02]
Presolving model
1 rows, 63 cols, 63 nonzeros  0s
1 rows, 14 cols, 14 nonzeros  0s
Dependent equations search running on 1 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
1 rows, 14 cols, 14 nonzeros  0s
Presolve reductions: rows 1(-98); columns 14(-72); nonzeros 14(-170) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
          1    -4.3396463771e+06 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-vjublopw
Model status        : Optimal
Simplex   iterations: 1


('ok', 'optimal')

#### Model 2. Reserve market clearing

In the reserve market, we assume that the generators calculated their offer price as described in the previous section. The market clears by minimising the cost of reserve.

In [142]:
# Calculate reserves offers 
expected_market_price = model.constraints['power_balance'].dual.values

piup = np.maximum(0, expected_market_price - ci) + 0.03 * ci
pidown = np.maximum(0, ci - expected_market_price) + 0.03 * ci

rup_max = gen_constraints_xr['Rplus_MW']
rdown_max = gen_constraints_xr['Rminus_MW']

In [143]:
# Reserve needs
need_for_up_reserve = 0.15
need_for_down_reserve = 0.1

In [144]:
up_reserve = demand_bid_xr['Quantity_MW'].sum().values*need_for_up_reserve
down_reserve = demand_bid_xr['Quantity_MW'].sum().values*need_for_down_reserve

In [145]:
r_model = linopy.Model()

# Decision variables: reserves
rup = r_model.add_variables(
    coords=[generators_set],
    name="rup"
)

r_down = r_model.add_variables(
    coords=[generators_set],
    name="rdown"
)

# Reserve capacity constraints
rup_constraints = r_model.add_constraints(
    rup <= rup_max,
    name="rup_capacity_limit"
)

rdown_constraints = r_model.add_constraints(
    r_down <= rdown_max,
    name="rdown_capacity_limit"
)

min_rup_constraints = r_model.add_constraints(
    rup >= 0,
    name="min_rup_capacity_limit"
)

min_rdown_constraints = r_model.add_constraints(
    r_down >= 0,
    name="min_rdown_capacity_limit"
)

# Pmax reserve constraint
pmax_reserve_constraints = r_model.add_constraints(
    rup + r_down <= pmax,
    name="pmax_reserve_limit"
)


# Reserve balance constraints
reserve_balance_up = r_model.add_constraints(
    rup.sum(dim="Unit") >= up_reserve,
    name="reserve_balance_up"
)

reserve_balance_down = r_model.add_constraints(
    r_down.sum(dim="Unit") >= down_reserve,
    name="reserve_balance_down"
)

# Then use these in your objective
r_model.add_objective(
    (rup * piup).sum() + (r_down * pidown).sum(),
    sense="min"
)

r_model.solve(solver_name="highs")

Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-ti9wru2l has 62 rows; 24 cols; 96 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [2e-01, 2e+01]
  Bound   [0e+00, 0e+00]
  RHS     [3e+01, 6e+02]
Presolving model
3 rows, 7 cols, 9 nonzeros  0s
Dependent equations search running on 1 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
3 rows, 6 cols, 8 nonzeros  0s
Presolve reductions: rows 3(-59); columns 6(-18); nonzeros 8(-88) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     1.3944600000e+02 Pr: 2(288.873) 0.0s
          2     6.5016545075e+02 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-ti9wru2l
Model status        : Optimal
Simplex   iterations: 2
Objective value     :  6.5016545075e

('ok', 'optimal')

#### Model 3. Market clearing for Day Ahead Market

Reserves found are now used as constraints in the day ahead market.

Here we are assigning minimums for generators, but in reality they would have to offer these minimums at a very cheap price to make sure they are dispatched and then be able to down regulate if needed. 

In [146]:
rup_solution = rup.solution
rdown_solution = r_down.solution

In [147]:
model2 = linopy.Model()

# Decision variables: generation
g = model2.add_variables(
    coords=[generators_set],
    name="dispatch"
)

# Decision variables: wind generation
w = model2.add_variables(
    lower=0,
    coords=[wind_farms_set],
    name="wind_generation"
)

# Decision variables: demand
d = model2.add_variables(
    lower=0,
    coords=[loads_set, bids_set],
    name="demand"
)

# Generators capacity constraints
capacity_constraints = model2.add_constraints(
    g <= pmax - rup_solution,  # Adjusted for up reserve
    name="capacity_limit"
)

min_generation_constraint = model2.add_constraints(
    g >= rdown_solution,  # Adjusted for down reserve
    name="min_capacity_limit"
)

# Wind generation constraints
wind_constraints = model2.add_constraints(
    w<= wind_forecast,
    name="wind_dispatch"
)

# Demand constraints
demand_constraints = model2.add_constraints(
    d <= demand_quantity,
    name="demand_limit"
)

# Power balance constraint
balance = model2.add_constraints(
    g.sum(dim="Unit") + w.sum(dim="Unit") == d.sum(dim=["Load","Bid"]),
    name="power_balance"
)

# Then use these in your objective
model2.add_objective(
    (g * ci).sum() - (d * demand_price).sum(),
    sense="min"
)

model2.solve(solver_name="highs")

Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-oyr046b9 has 99 rows; 86 cols; 184 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [5e+00, 2e+03]
  Bound   [0e+00, 0e+00]
  RHS     [3e+00, 4e+02]
Presolving model
1 rows, 63 cols, 63 nonzeros  0s
1 rows, 14 cols, 14 nonzeros  0s
Dependent equations search running on 1 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
1 rows, 14 cols, 14 nonzeros  0s
Presolve reductions: rows 1(-98); columns 14(-72); nonzeros 14(-170) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
          1    -4.3392945087e+06 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-oyr046b9
Model status        : Optimal
Simplex   iterations: 1


('ok', 'optimal')

#### Model 4. Joint reserves market clearing

In this model we clear the day ahead market and the reserve market together, to compare US style vs EU style market clearing.

In [148]:
joint_model = linopy.Model()

# Decision variables: generation
g = joint_model.add_variables(
    coords=[generators_set],
    name="dispatch"
)

# Decision variables: wind generation
w = joint_model.add_variables(
    lower=0,
    coords=[wind_farms_set],
    name="wind_generation"
)

# Decision variables: demand
d = joint_model.add_variables(
    lower=0,
    coords=[loads_set, bids_set],
    name="demand"
)

# Decision variables: reserves
rup = joint_model.add_variables(
    coords=[generators_set],
    name="rup"
)

r_down = joint_model.add_variables(
    coords=[generators_set],
    name="rdown"
)

# Generators capacity constraints
capacity_constraints = joint_model.add_constraints(
    g + rup <= pmax,  # Adjusted for up reserve
    name="capacity_limit"
)

min_generation_constraint = joint_model.add_constraints(
    g - r_down >= 0,  # Adjusted for down reserve
    name="min_capacity_limit"
)

# Wind generation constraints
wind_constraints = joint_model.add_constraints(
    w<= wind_forecast,
    name="wind_dispatch"
)

# Demand constraints
demand_constraints = joint_model.add_constraints(
    d <= demand_quantity,
    name="demand_limit"
)

# Reserve max capacity constraints
rup_constraints = joint_model.add_constraints(
    rup <= rup_max,
    name="rup_capacity_limit"
)

rdown_constraints = joint_model.add_constraints(
    r_down <= rdown_max,
    name="rdown_capacity_limit"
)

min_rup_constraints = joint_model.add_constraints(
    rup >= 0,
    name="min_rup_capacity_limit"
)

min_rdown_constraints = joint_model.add_constraints(
    r_down >= 0,
    name="min_rdown_capacity_limit"
)

# Reserve balance constraints
reserve_balance_up = joint_model.add_constraints(
    rup.sum(dim="Unit") >= up_reserve,
    name="reserve_balance_up"
)

reserve_balance_down = joint_model.add_constraints(
    r_down.sum(dim="Unit") >= down_reserve,
    name="reserve_balance_down"
)


# Power balance constraint
balance = joint_model.add_constraints(
    g.sum(dim="Unit") + w.sum(dim="Unit") == d.sum(dim=["Load","Bid"]),
    name="power_balance"
)

# Objective function
joint_model.add_objective(
    (g * ci).sum() - (d * demand_price).sum(),
    sense="min"
)

joint_model.solve(solver_name="highs")


Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-ukz9q9m_ has 149 rows; 110 cols; 280 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [5e+00, 2e+03]
  Bound   [0e+00, 0e+00]
  RHS     [3e+00, 6e+02]
Presolving model
27 rows, 81 cols, 123 nonzeros  0s
21 rows, 35 cols, 71 nonzeros  0s
Dependent equations search running on 1 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
21 rows, 35 cols, 71 nonzeros  0s
Presolve reductions: rows 21(-128); columns 35(-75); nonzeros 71(-209) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -1.3683000000e+05 Ph1: 19(26991); Du: 9(136.83) 0.0s
         15    -4.3392994386e+06 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-ukz9q9m_
Model status        : Opti

('ok', 'optimal')

### Results and Analysis

#### Reserve market

This section analyses the results of the reserve market clearing.

In [149]:
# Recall the situation

print("="*50)
print(f"Need for up reserve: {up_reserve:.2f} MW")
print(f"Need for down reserve: {down_reserve:.2f} MW")
print("="*50 + "\n")

print("Reserve Offers:")
reserve_offer_df = pd.DataFrame({
    'Unit': generators_set.values,
    'Up Reserve Offer ($/MW)': piup.values,
    'Down Reserve Offer ($/MW)': pidown.values,
    'Up Max Capacity (MW)': rup_max.values,
    'Down Max Capacity (MW)': rdown_max.values,
})
reserve_offer_df


Need for up reserve: 383.32 MW
Need for down reserve: 255.55 MW

Reserve Offers:


,Unit,Up Reserve Offer ($/MW),Down Reserve Offer ($/MW),Up Max Capacity (MW),Down Max Capacity (MW)
0,G1,0.3996,2.8296,40,40
1,G2,0.3996,2.8296,40,40
2,G3,0.6210,10.4310,70,70
3,G4,0.6279,10.6679,180,180
4,G5,0.7833,16.0033,60,60
5,G6,0.6856,0.3156,30,30
6,G7,0.6856,0.3156,30,30
7,G8,5.0506,0.1806,0,0
8,G9,5.5841,0.1641,0,0
9,G10,10.8900,0.0000,0,0


The prices offered by generators are linked to their opportunity cost, but it has a risk component as the day ahead price is uncertain. 

In [162]:
# Results of the reserve market clearing

r_up_price = r_model.constraints['reserve_balance_up'].dual.values
r_down_price = r_model.constraints['reserve_balance_down'].dual.values

day_ahead_payment_no_reserve = (model.variables['dispatch'].solution.sum() + model.variables['wind_generation'].solution.sum())*expected_market_price
reserve_cost = rup_solution.sum()*r_up_price + rdown_solution.sum()*r_down_price

print("="*50)
print("Price for up reserve: ${:.2f}/MW".format(r_up_price))
print("Price for down reserve: ${:.2f}/MW".format(r_down_price))
print("Up reserve cost: ${:,.2f}".format(rup_solution.sum()*r_up_price))
print("Down reserve cost: ${:,.2f}".format(rdown_solution.sum()*r_down_price))
print("="*50 + "\n")

Price for up reserve: $0.69/MW
Price for down reserve: $10.43/MW
Up reserve cost: $262.81
Down reserve cost: $2,665.63



In [163]:
# Results per generator
rup_result = rup_solution.to_dataframe().rename(columns={'solution': 'Rup_MW'})
rdown_result = rdown_solution.to_dataframe().rename(columns={'solution': 'Rdown_MW'})
reserve_df = pd.concat([rup_result, rdown_result], axis=1)
reserve_df['Rup_Payment_$'] = reserve_df['Rup_MW'] * r_up_price
reserve_df['Rdown_Payment_$'] = reserve_df['Rdown_MW'] * r_down_price
reserve_df['Total_Payment_$'] = reserve_df['Rup_Payment_$'] + reserve_df['Rdown_Payment_$']
reserve_df.round(2)

,Rup_MW,Rdown_MW,Rup_Payment_$,Rdown_Payment_$,Total_Payment_$
Unit,,,,,
G1,40.00,40.00,27.42,417.24,444.66
G2,40.00,40.00,27.42,417.24,444.66
G3,70.00,15.55,47.99,162.19,210.19
G4,180.00,-0.00,123.41,-0.00,123.41
G5,-0.00,-0.00,-0.00,-0.00,-0.00
G6,-0.00,30.00,-0.00,312.93,312.93
G7,-0.00,30.00,-0.00,312.93,312.93
G8,-0.00,-0.00,-0.00,-0.00,-0.00
G9,-0.00,-0.00,-0.00,-0.00,-0.00


It is observed that the up rerserve price is much higher than the down reserve price. This is because for up regulation, all generators that are not at their maximum are indifferent between offering up reserve as they would have that capacity available anyways. However, for down regulation, only generators that are expected to be dispatched would be offering down reserve at a very lw cost, but those that are not expected to be dispatched would have to offer down reserve at a much higher price, as they would have to operate at a higher point of their cost curve to be able to offer down reserve.

Hence, although there is more need for up reserve than down reserve, the cost of down reserve is higher as it would involve some generators to operate at a loss in the day ahead market.

#### Change in day ahead market

Reserves will change the dispatch in the day ahead market, then the cost of generation and the market price will also change. In this section we will look at how the day ahead market changes when we include reserve requirements.

In [174]:
# Results comparisons

dam_price = model2.constraints['power_balance'].dual.values
day_ahead_payment_after_reserve = (model2.variables['dispatch'].solution.sum() + model2.variables['wind_generation'].solution.sum())*dam_price
total_change_in_payment = day_ahead_payment_after_reserve - day_ahead_payment_no_reserve + reserve_cost


print("="*50)
print("Scenario with no reserves:\n")
print(f"DAM price (expected): ${expected_market_price:.2f}/MWh")
print(f"DAM payment (expected): ${day_ahead_payment_no_reserve:,.2f}")
print("="*50 + "\n")
print("="*50)
print("Scenario with reserves:\n")
print(f"DAM price after reserve: ${dam_price:.2f}/MWh")
print(f"DAM payment after reserve: ${day_ahead_payment_after_reserve:,.2f}")
print(f"Reserve cost: ${reserve_cost:,.2f}")
print("="*50 + "\n")

print("="*50)
print("Comparisons\n")
print(f"Total change in payment due to reserve: ${total_change_in_payment:,.2f}")
print(f"Total change in payment as percentage of day ahead payment without reserve: {total_change_in_payment/day_ahead_payment_no_reserve*100:.2f}%")
print("="*50 + "\n")

# Reserve payment, day ahead payment, cost, profit for each generator and wind farm
gen_result = model2.variables['dispatch'].solution.to_dataframe().rename(columns={'solution': 'Generation_MW'})
wind_result = model2.variables['wind_generation'].solution.to_dataframe().rename(columns={'solution': 'Generation_MW'})
gen_result['Generation_Cost_$'] = gen_result['Generation_MW'] * ci.values
gen_result['Up_Reserve_Pmt_$'] = rup_solution.values * r_up_price
gen_result['Down_Reserve_Pmt_$'] = rdown_solution.values * r_down_price
all_gen_result = pd.concat([gen_result, wind_result], axis=0).fillna(0)
all_gen_result['DA_Pmt_$'] = all_gen_result['Generation_MW'] * dam_price
all_gen_result['DA_Profit_$'] = all_gen_result['DA_Pmt_$'] - all_gen_result['Generation_Cost_$']
all_gen_result['Total_Profit_$'] = all_gen_result['DA_Profit_$'] + all_gen_result['Up_Reserve_Pmt_$'] + all_gen_result['Down_Reserve_Pmt_$']

# join previous
gen_previous = model.variables['dispatch'].solution.to_dataframe().rename(columns={'solution': 'Expected_MW'})
gen_previous['Ci$'] =ci.values
wind_previous = model.variables['wind_generation'].solution.to_dataframe().rename(columns={'solution': 'Expected_MW'})
previous = pd.concat([gen_previous, wind_previous], axis=0).fillna(0)
previous['Expected_Profit_$'] = previous['Expected_MW'] * expected_market_price - previous['Expected_MW'] * previous['Ci$'].values

all_gen_result['Diff_no_reserve_$'] = all_gen_result['Total_Profit_$'] - previous['Expected_Profit_$']

print("Results per generator:")
all_gen_result.round(2)


Scenario with no reserves:

DAM price (expected): $10.89/MWh
DAM payment (expected): $27,420.75

Scenario with reserves:

DAM price after reserve: $10.89/MWh
DAM payment after reserve: $27,420.75
Reserve cost: $2,928.44

Comparisons

Total change in payment due to reserve: $2,928.44
Total change in payment as percentage of day ahead payment without reserve: 10.68%

Results per generator:


,Generation_MW,Generation_Cost_$,Up_Reserve_Pmt_$,Down_Reserve_Pmt_$,DA_Pmt_$,DA_Profit_$,Total_Profit_$,Diff_no_reserve_$
Unit,,,,,,,,
G1,40.00,532.80,27.42,417.24,435.60,-97.20,347.46,347.46
G2,40.00,532.80,27.42,417.24,435.60,-97.20,347.46,347.46
G3,15.55,321.87,47.99,162.19,169.33,-152.54,57.65,57.65
G4,-0.00,-0.00,123.41,-0.00,-0.00,0.00,123.41,123.41
G5,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00,0.00
G6,155.00,1630.60,-0.00,312.93,1687.95,57.35,370.28,312.93
G7,155.00,1630.60,-0.00,312.93,1687.95,57.35,370.28,312.93
G8,400.00,2408.00,-0.00,-0.00,4356.00,1948.00,1948.00,0.00
G9,400.00,2188.00,-0.00,-0.00,4356.00,2168.00,2168.00,0.00


In this scenario, the reserve market didn't change the day ahead market price, but it made some generators to be dispatched at a value lower than their variable cost, so they were operating at a loss. However they recovered that loss in the reserve market. This couls have been different if the expected market price was higher or lower, which is a risk that the generators take when they offer reserve, and that's why we would expect them to add a premium to their offer price for reserve.

By changing the reserve need or the risk premium, there are changes in the day ahead market price, that makes some generators to have more profits while others might have losses. 

#### Comparison of market clearing approaches

In [175]:
jm_dam_price = joint_model.constraints['power_balance'].dual.values
jm_rup_price = joint_model.constraints['reserve_balance_up'].dual.values
jm_rdown_price = joint_model.constraints['reserve_balance_down'].dual.values

print("="*50)
print(f"Joint model DAM price: ${jm_dam_price:.2f}/MWh")
print(f"Joint model up reserve price: ${jm_rup_price:.2f}/MW")
print(f"Joint model down reserve price: ${jm_rdown_price:.2f}/MW")
print("="*50 + "\n")


jm_up_reserve_cost = joint_model.variables['rup'].solution.sum() * jm_rup_price
jm_down_reserve_cost = joint_model.variables['rdown'].solution.sum() * jm_rdown_price
jm_total_reserve_cost = jm_up_reserve_cost + jm_down_reserve_cost
jm_dam_payments = (joint_model.variables['dispatch'].solution.sum() + joint_model.variables['wind_generation'].solution.sum()) * jm_dam_price
jm_total_cost = jm_dam_payments + jm_total_reserve_cost
print("="*50)
print(f"Joint model up reserve cost: ${jm_up_reserve_cost:,.2f}")
print(f"Joint model down reserve cost: ${jm_down_reserve_cost:,.2f}")
print(f"Joint model total reserve cost: ${jm_total_reserve_cost:,.2f}")
print(f"Joint model day ahead payments: ${jm_dam_payments:,.2f}")
print("="*50 + "\n")

print("="*50)
print(f"Total cost sequential optimization: ${day_ahead_payment_after_reserve + reserve_cost:,.2f}")
print(f"Total cost joint optimization: ${jm_total_cost:,.2f}\n")
print(f"Difference in total cost: ${day_ahead_payment_after_reserve + reserve_cost - jm_total_cost:,.2f}")
print(f"Difference in reserve cost: ${reserve_cost - jm_total_reserve_cost:,.2f}")
print(f"Difference in day ahead payments: ${day_ahead_payment_after_reserve - jm_dam_payments:,.2f}")
print("="*50 )

Joint model DAM price: $10.89/MWh
Joint model up reserve price: $-0.00/MW
Joint model down reserve price: $9.81/MW

Joint model up reserve cost: $-0.00
Joint model down reserve cost: $2,506.94
Joint model total reserve cost: $2,506.94
Joint model day ahead payments: $27,420.75

Total cost sequential optimization: $30,349.19
Total cost joint optimization: $29,927.69

Difference in total cost: $421.50
Difference in reserve cost: $421.50
Difference in day ahead payments: $0.00


There is a small difference in the results of the joint market clearing and the sequential market clearing, which is due to the risk premium added in the sequential market clearing. When clearing in the co-optimisation, the opportunity cost of generators is calculated endogenously, so there is no need to add a risk premium to their offer price for reserve. In fact, if we run the exact same model without the risk premium, we get the same results as the joint market clearing. However, this is not true under other conditions of reserve needs, where reserve might influence the day ahead market price and in those cases some generators might have losses or, on the contrary, more profits at expense of consumers.

In [ ]:
# Results per generator
jm_gen_result = joint_model.variables['dispatch'].solution.to_dataframe().rename(columns={'solution': 'Generation_MW'})
jm_wind_result = joint_model.variables['wind_generation'].solution.to_dataframe().rename(columns={'solution': 'Generation_MW'})
jm_gen_result['Generation_Cost_$'] = jm_gen_result['Generation_MW'] * ci.values
jm_gen_result['Up_Reserve_Pmt_$'] = joint_model.variables['rup'].solution.values * jm_rup_price
jm_gen_result['Down_Reserve_Pmt_$'] = joint_model.variables['rdown'].solution.values * jm_rdown_price
jm_all_gen_result = pd.concat([jm_gen_result, jm_wind_result], axis=0).fillna(0)
jm_all_gen_result['DA_Pmt_$'] = jm_all_gen_result['Generation_MW'] * jm_dam_price
jm_all_gen_result['DA_Profit_$'] = jm_all_gen_result['DA_Pmt_$'] - jm_all_gen_result['Generation_Cost_$']
jm_all_gen_result['Total_Profit_$'] = jm_all_gen_result['DA_Profit_$'] + jm_all_gen_result['Up_Reserve_Pmt_$'] + jm_all_gen_result['Down_Reserve_Pmt_$']
jm_all_gen_result['Diff_no_reserve_$'] = jm_all_gen_result['Total_Profit_$'] - previous['Expected_Profit_$']

print("Joint model results per generator:")
jm_all_gen_result.round(2)

Joint model results per generator:


,Generation_MW,Generation_Cost_$,Up_Reserve_Pmt_$,Down_Reserve_Pmt_$,DA_Pmt_$,DA_Profit_$,Total_Profit_$,Diff_no_reserve_$
Unit,,,,,,,,
G1,40.00,532.80,0.0,392.40,435.60,-97.20,295.20,295.2
G2,40.00,532.80,-0.0,392.40,435.60,-97.20,295.20,295.2
G3,15.55,321.87,-0.0,152.54,169.33,-152.54,0.00,0.0
G4,-0.00,-0.00,-0.0,-0.00,-0.00,0.00,0.00,0.0
G5,-0.00,-0.00,-0.0,-0.00,-0.00,0.00,0.00,0.0
G6,155.00,1630.60,0.0,294.30,1687.95,57.35,351.65,294.3
G7,155.00,1630.60,0.0,294.30,1687.95,57.35,351.65,294.3
G8,400.00,2408.00,0.0,-0.00,4356.00,1948.00,1948.00,0.0
G9,400.00,2188.00,0.0,-0.00,4356.00,2168.00,2168.00,0.0


In [155]:
# Profit per generator in Sequential vs Joint optimization
comparison_df = pd.DataFrame({
    'Sequential_Total_Profit_$': all_gen_result['Total_Profit_$'],
    'Joint_Total_Profit_$': jm_all_gen_result['Total_Profit_$']
})
comparison_df['Profit_Difference_$'] = comparison_df['Joint_Total_Profit_$'] - comparison_df['Sequential_Total_Profit_$']
print("Comparison of total profit per generator between sequential and joint optimization:")
comparison_df.round(2)


Comparison of total profit per generator between sequential and joint optimization:


,Sequential_Total_Profit_$,Joint_Total_Profit_$,Profit_Difference_$
Unit,,,
G1,347.46,295.20,-52.26
G2,347.46,295.20,-52.26
G3,57.65,0.00,-57.65
G4,123.41,0.00,-123.41
G5,0.00,0.00,0.00
G6,370.28,351.65,-18.63
G7,370.28,351.65,-18.63
G8,1948.00,1948.00,0.00
G9,2168.00,2168.00,0.00


##### Lagrange analysis

<!-- Let's define the dual variables asociated to the min and max capacity constraints as $\mu_i^{max}$ and $\mu_i^{min}$, respectively. Then the dual variables associated to the reserve up and down constraints are $\pi_i^{up}$ and $\pi_i^{down}$, respectively. Then, when taking the Lagrangian of the joint optimization problem with respect to the generation variable, we get: -->
- Power constraint dual variables: $\overline{\mu}_i$, $\underline{\mu}_i$
- Reserve up constraint dual variables: $\overline{\pi_i}^{up}$, $\underline{\pi_i}^{up}$
- Reserve down constraint dual variables: $\overline{\pi_i}^{down}$, $\underline{\pi_i}^{down}$
- Power balance constraint dual variable: $\lambda$
- Reserve up balance constraint dual variable: $\lambda^{up}$
- Reserve down balance constraint dual variable: $\lambda^{down}$

The Lagrangian of the joint optimization problem is:
$$\mathcal{L} = (\sum_i c_i g_i - \sum_j p_j D_j)\\
+ \sum_i \overline{\mu}_i (g_i + r_i^{up} - G_i^{max}) - \sum_i \underline{\mu}_i (g_i - r_i^{down})\\
+ \sum_i \overline{\pi_i}^{up} (r_i^{up} - R_{up}^{max}) - \sum_i \underline{\pi_i}^{up} (r_i^{up})\\
+ \sum_i \overline{\pi_i}^{down} (r_i^{down} - R_{down}^{max}) - \sum_i \underline{\pi_i}^{down} (r_i^{down})\\
- \lambda (\sum_i g_i - \sum_j D_j)\\
- \lambda^{up} (\sum_i r_i^{up} - R_{up})\\
- \lambda^{down} (\sum_i r_i^{down} - R_{down})$$

Taking the derivative of the Lagrangian with respect to the generation variable $g_i$, we get:
$$\frac{\partial \mathcal{L}}{\partial g_i} = c_i + \overline{\mu}_i - \underline{\mu}_i - \lambda = 0$$

Taking the derivative of the Lagrangian with respect to the reserve up variable $r_i^{up}$, we get:
$$\frac{\partial \mathcal{L}}{\partial r_i^{up}} = \overline{\mu}_i + \overline{\pi_i}^{up} - \underline{\pi_i}^{up} - \lambda^{up} = 0$$

Taking the derivative of the Lagrangian with respect to the reserve down variable $r_i^{down}$, we get:
$$\frac{\partial \mathcal{L}}{\partial r_i^{down}} = \underline{\mu}_i + \overline{\pi_i}^{down} - \underline{\pi_i}^{down} - \lambda^{down} = 0$$

The variables $\overline{\mu}_i$ and $\underline{\mu}_i$ are the opportunity costs of generation.

In [156]:
jm_mu_max = joint_model.constraints['capacity_limit'].dual
jm_mu_min = joint_model.constraints['min_capacity_limit'].dual
jm_pi_up_max = joint_model.constraints['rup_capacity_limit'].dual
jm_pi_up_min = joint_model.constraints['min_rup_capacity_limit'].dual
jm_pi_down_max = joint_model.constraints['rdown_capacity_limit'].dual
jm_pi_down_min = joint_model.constraints['min_rdown_capacity_limit'].dual

In [158]:
pd.DataFrame({
    'unit': generators_set,
    'ci_$/MWh': ci.values,
    'mu_max': jm_mu_max,
    'mu_min': jm_mu_min,
    'pi_up_max': jm_pi_up_max,
    'pi_up_min': jm_pi_up_min,
    'pi_down_max': jm_pi_down_max,
    'pi_down_min': jm_pi_down_min,
    'lambda_up': jm_rup_price,
    'lambda_down': jm_rdown_price,
    'lambda_dam': jm_dam_price
})

,unit,ci_$/MWh,mu_max,mu_min,pi_up_max,pi_up_min,pi_down_max,pi_down_min,lambda_up,lambda_down,lambda_dam
0,G1,13.32,-0.00,2.43,-0.0,-0.00,-7.38,-0.00,-0.0,9.81,10.89
1,G2,13.32,-0.00,2.43,-0.0,-0.00,-7.38,-0.00,-0.0,9.81,10.89
2,G3,20.70,-0.00,9.81,-0.0,-0.00,-0.00,-0.00,-0.0,9.81,10.89
3,G4,20.93,-0.00,10.04,-0.0,-0.00,-0.00,0.23,-0.0,9.81,10.89
4,G5,26.11,-0.00,15.22,-0.0,-0.00,-0.00,5.41,-0.0,9.81,10.89
5,G6,10.52,-0.37,-0.00,-0.0,0.37,-9.81,-0.00,-0.0,9.81,10.89
6,G7,10.52,-0.37,-0.00,-0.0,0.37,-9.81,-0.00,-0.0,9.81,10.89
7,G8,6.02,-4.87,-0.00,-0.0,4.87,-9.81,-0.00,-0.0,9.81,10.89
8,G9,5.47,-5.42,-0.00,-0.0,5.42,-9.81,-0.00,-0.0,9.81,10.89
9,G10,0.00,-10.89,-0.00,-0.0,10.89,-9.81,-0.00,-0.0,9.81,10.89
